# Stage 6 — Generative Inverse-Modeling (Circuit Decoder)

**Goal:** Train a decoder that reconstructs input images from circuit embeddings z, then
use it to visualise what the model's internal reasoning actually encodes.

**Core question:** Does the CTLS circuit space contain enough visual information to
reconstruct images — and does it contain *more* than the unstructured baseline space?

**Four experiments:**
1. **Reconstruction quality** — side-by-side: original | CTLS recon | baseline recon
2. **Centroid decoding** — decode the mean z per class → the model's 'Universal Dog', etc.
3. **Latent interpolation** — decode steps between two class centroids → visual ambiguity zone
4. **Confusion zone** — find class-A images closest to class-B's centroid → what drives confusion

**Key metric:** Val MSE: CTLS decoder should achieve lower error than baseline decoder,
confirming that structured z encodes more recoverable visual information.

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
        os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f"{DRIVE_BASE}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/experiments", exist_ok=True)
    os.makedirs(f"{REPO_DIR}/data", exist_ok=True)

    for link, target in [
        (f"{REPO_DIR}/data/raw",    f"{DRIVE_BASE}/data/raw"),
        (f"{REPO_DIR}/experiments", f"{DRIVE_BASE}/experiments"),
    ]:
        if os.path.islink(link): os.unlink(link)
        os.symlink(target, link)

    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()} ({'Colab' if IN_COLAB else 'local'})")

In [ ]:
import yaml

with open('configs/decoder.yaml') as f:
    config = yaml.safe_load(f)

print(yaml.dump(config, default_flow_style=False))

## 1. Train Decoders

Two decoders are trained post-hoc with frozen backbones:
- **CTLS decoder** — trained on embeddings from `experiments/ctls/best.pt`
- **Baseline decoder** — trained on embeddings from `experiments/baseline/best.pt`

Training takes ~10–15 min on a single GPU (50 epochs, batch size 256).

**Prerequisites:** `experiments/ctls/best.pt` and `experiments/baseline/best.pt`
must exist. Run stages 1 and 2 first if they don't.

**Skip if `experiments/decoder/decoder_ctls_best.pt` already exists.**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from pathlib import Path

from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from models.decoder import CircuitDecoder
from data.cifar import get_standard_loaders

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

dcfg = config['data']
train_loader, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'],
    batch_size=dcfg['batch_size'],
    num_workers=dcfg.get('num_workers', 4),
    augment=dcfg.get('augment', True),
    download=True,
)

In [ ]:
def load_frozen_encoder(config, checkpoint_path, device):
    """Load backbone + meta-encoder, freeze both."""
    mcfg = config['model']
    ecfg = mcfg['meta_encoder']
    sm_temp = mcfg['soft_mask']['init_temperature']

    soft_mask = SoftMask(init_temperature=sm_temp)
    backbone = CTLSBackbone(
        arch=mcfg['arch'],
        num_classes=mcfg['num_classes'],
        soft_mask=soft_mask,
        pretrained=False,
    ).to(device)
    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        hidden_dim=ecfg.get('hidden_dim', 256),
        embedding_dim=ecfg.get('embedding_dim', 64),
        encoder_type=ecfg.get('encoder_type', 'mlp'),
    ).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device)
    backbone.load_state_dict(ckpt['backbone_state'])
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    print(f"  Loaded: epoch={ckpt['epoch']}, val_acc={ckpt['val_acc']:.4f}")

    backbone.eval()
    meta_encoder.eval()
    for p in backbone.parameters():    p.requires_grad_(False)
    for p in meta_encoder.parameters(): p.requires_grad_(False)

    return backbone, meta_encoder


def train_decoder_inline(backbone, meta_encoder, train_loader, val_loader,
                          config, save_dir, tag, device):
    tcfg = config['training']
    dcfg_d = config['decoder']
    lcfg = config['logging']

    decoder = CircuitDecoder(
        embedding_dim=dcfg_d.get('embedding_dim', 64),
        hidden_channels=dcfg_d.get('hidden_channels', [256, 128, 64, 32]),
        input_spatial=dcfg_d.get('input_spatial', 4),
    ).to(device)

    epochs = tcfg['epochs']
    optimizer = AdamW(decoder.parameters(),
                      lr=float(tcfg.get('lr', 1e-3)),
                      weight_decay=float(tcfg.get('weight_decay', 1e-4)))
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    log_interval = lcfg.get('log_interval', 50)
    save_every   = lcfg.get('save_every', 10)
    best_val_mse = float('inf')

    for epoch in range(epochs):
        decoder.train()
        total, n_b = 0.0, 0
        for batch_idx, (x, _) in enumerate(train_loader):
            x = x.to(device)
            with torch.no_grad():
                _, traj = backbone(x)
                z = meta_encoder(traj)
            x_hat = decoder(z)
            loss  = F.mse_loss(x_hat, x)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
            optimizer.step()
            total += loss.item(); n_b += 1
            if (batch_idx + 1) % log_interval == 0:
                print(f"  [{tag}] [{batch_idx+1}/{len(train_loader)}] mse={loss.item():.5f}")

        scheduler.step()

        decoder.eval()
        val_loss, val_n = 0.0, 0
        with torch.no_grad():
            for x, _ in val_loader:
                x = x.to(device)
                _, traj = backbone(x)
                z = meta_encoder(traj)
                val_loss += F.mse_loss(decoder(z), x).item()
                val_n += 1

        avg_train = total / n_b
        avg_val   = val_loss / val_n
        print(f"Epoch {epoch+1:3d}/{epochs} [{tag}] | "
              f"train_mse={avg_train:.5f}  val_mse={avg_val:.5f}")

        if avg_val < best_val_mse:
            best_val_mse = avg_val
            torch.save({'epoch': epoch, 'val_mse': avg_val,
                        'decoder_state': decoder.state_dict(), 'config': config},
                       save_dir / f'decoder_{tag}_best.pt')

        if (epoch + 1) % save_every == 0:
            torch.save({'epoch': epoch, 'val_mse': avg_val,
                        'decoder_state': decoder.state_dict(), 'config': config},
                       save_dir / f'decoder_{tag}_epoch_{epoch+1}.pt')

    print(f"[{tag}] Best val MSE: {best_val_mse:.5f}")
    return decoder, best_val_mse

In [ ]:
save_dir = Path('experiments/decoder')
save_dir.mkdir(parents=True, exist_ok=True)

print('=== Training CTLS decoder ===')
backbone_ctls, meta_enc_ctls = load_frozen_encoder(
    config, config['training']['ctls_checkpoint'], DEVICE
)
decoder_ctls, mse_ctls = train_decoder_inline(
    backbone_ctls, meta_enc_ctls, train_loader, val_loader,
    config, save_dir, tag='ctls', device=DEVICE,
)

print('\n=== Training Baseline decoder ===')
with open('configs/baseline.yaml') as f:
    base_config = yaml.safe_load(f)
# Use decoder config's soft_mask temperature, not baseline's
base_config['model']['soft_mask'] = config['model']['soft_mask']

backbone_base, meta_enc_base = load_frozen_encoder(
    base_config, config['training']['baseline_checkpoint'], DEVICE
)
decoder_base, mse_base = train_decoder_inline(
    backbone_base, meta_enc_base, train_loader, val_loader,
    config, save_dir, tag='baseline', device=DEVICE,
)

print(f"\nCTLS val MSE:     {mse_ctls:.5f}")
print(f"Baseline val MSE: {mse_base:.5f}")
print(f"Delta:            {mse_ctls - mse_base:+.5f}  (negative = CTLS reconstructs better)")

## 2. Load Checkpoints

If you already have trained decoders, start here.

In [ ]:
import yaml
import torch
from pathlib import Path
from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from models.decoder import CircuitDecoder
from data.cifar import get_standard_loaders

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('configs/decoder.yaml') as f:
    config = yaml.safe_load(f)


def load_encoder_and_decoder(enc_config, dec_config, enc_ckpt_path, dec_ckpt_path, device):
    """
    enc_config: config dict that has 'model' (arch, meta_encoder, soft_mask)
    dec_config: config dict that has 'decoder' (embedding_dim, hidden_channels, input_spatial)
                — always pass the main decoder config here, regardless of which encoder is loaded
    """
    mcfg = enc_config['model']
    ecfg = mcfg['meta_encoder']
    dcfg = dec_config['decoder']

    soft_mask = SoftMask(init_temperature=mcfg['soft_mask']['init_temperature'])
    backbone = CTLSBackbone(
        arch=mcfg['arch'], num_classes=mcfg['num_classes'],
        soft_mask=soft_mask, pretrained=False,
    ).to(device)
    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        hidden_dim=ecfg.get('hidden_dim', 256),
        embedding_dim=ecfg.get('embedding_dim', 64),
        encoder_type=ecfg.get('encoder_type', 'mlp'),
    ).to(device)

    enc_ckpt = torch.load(enc_ckpt_path, map_location=device)
    backbone.load_state_dict(enc_ckpt['backbone_state'])
    meta_encoder.load_state_dict(enc_ckpt['meta_encoder_state'])
    backbone.eval(); meta_encoder.eval()

    decoder = CircuitDecoder(
        embedding_dim=dcfg.get('embedding_dim', 64),
        hidden_channels=dcfg.get('hidden_channels', [256, 128, 64, 32]),
        input_spatial=dcfg.get('input_spatial', 4),
    ).to(device)
    dec_ckpt = torch.load(dec_ckpt_path, map_location=device)
    decoder.load_state_dict(dec_ckpt['decoder_state'])
    decoder.eval()

    print(f"  Decoder val MSE: {dec_ckpt['val_mse']:.5f}")
    return backbone, meta_encoder, decoder


print('Loading CTLS encoder + decoder...')
backbone_ctls, meta_enc_ctls, decoder_ctls = load_encoder_and_decoder(
    enc_config=config,
    dec_config=config,
    enc_ckpt_path='experiments/ctls/best.pt',
    dec_ckpt_path='experiments/decoder/decoder_ctls_best.pt',
    device=DEVICE,
)

print('Loading Baseline encoder + decoder...')
with open('configs/baseline.yaml') as f:
    base_config = yaml.safe_load(f)
base_config['model']['soft_mask'] = config['model']['soft_mask']

backbone_base, meta_enc_base, decoder_base = load_encoder_and_decoder(
    enc_config=base_config,
    dec_config=config,       # decoder architecture always from decoder.yaml
    enc_ckpt_path='experiments/baseline/best.pt',
    dec_ckpt_path='experiments/decoder/decoder_baseline_best.pt',
    device=DEVICE,
)

dcfg = config['data']
_, val_loader = get_standard_loaders(
    data_dir=dcfg['data_dir'],
    batch_size=128,
    num_workers=dcfg.get('num_workers', 4),
    augment=False,
    download=True,
)
print('Ready.')

## 3. Reconstruction Quality

Side-by-side: original | CTLS reconstruction | baseline reconstruction.

**What to look for:** CTLS reconstructions should be visibly closer to the
original — sharper class-relevant features, less noise. The baseline decoder
has less signal to work with (circuit silhouette 0.15 vs 0.81), so it should
produce blurrier or less class-coherent outputs.

In [ ]:
import matplotlib.pyplot as plt
from evaluation.decoder_viz import DecoderVisualizer

viz_ctls = DecoderVisualizer(backbone_ctls, meta_enc_ctls, decoder_ctls, val_loader, DEVICE)

fig = viz_ctls.reconstruction_grid(
    n_images=10,
    decoder_baseline=decoder_base,
    title='Stage 6 — Reconstruction Quality: CTLS vs Baseline',
)
os.makedirs('experiments/decoder', exist_ok=True)
fig.savefig('experiments/decoder/reconstruction_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Quantitative MSE Comparison

Per-class MSE for CTLS vs baseline decoders on the held-out validation set.

The CTLS decoder should achieve lower MSE across all (or most) classes,
confirming that structured circuit embeddings encode more recoverable visual information.

In [ ]:
viz_base = DecoderVisualizer(backbone_base, meta_enc_base, decoder_base, val_loader, DEVICE)

print('Computing per-class MSE...')
mse_ctls_cls  = viz_ctls.mse_by_class(max_samples=5000)
mse_base_cls  = viz_base.mse_by_class(max_samples=5000)

print()
print(f"{'Class':<14} {'CTLS MSE':>10} {'Base MSE':>10} {'Delta':>10}")
print('-' * 46)
for cls_name in mse_ctls_cls:
    c = mse_ctls_cls[cls_name]
    b = mse_base_cls[cls_name]
    print(f"{cls_name:<14} {c:>10.5f} {b:>10.5f} {c - b:>+10.5f}")
print('-' * 46)
mean_ctls = sum(mse_ctls_cls.values()) / 10
mean_base = sum(mse_base_cls.values()) / 10
print(f"{'Mean':<14} {mean_ctls:>10.5f} {mean_base:>10.5f} {mean_ctls - mean_base:>+10.5f}")

In [ ]:
# Bar chart: per-class MSE comparison
import numpy as np

classes = list(mse_ctls_cls.keys())
x = np.arange(len(classes))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - w/2, [mse_ctls_cls[c] for c in classes], w, label='CTLS decoder', color='darkorange')
ax.bar(x + w/2, [mse_base_cls[c] for c in classes], w, label='Baseline decoder', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=30, ha='right')
ax.set_ylabel('Reconstruction MSE (normalised pixel space)')
ax.set_title('Per-class reconstruction MSE: CTLS vs Baseline decoder')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('experiments/decoder/mse_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Centroid Decoding — "Universal X"

Compute the mean circuit embedding for each of the 10 CIFAR-10 classes,
then decode each centroid. The result shows what the model's circuits consider
the *prototypical* visual form of each class.

**What to look for:**
- CTLS centroids should produce sharper, more class-coherent images — the
  circuit embeddings form tighter clusters, so the centroid is more representative.
- Baseline centroids may be blurrier or noisier — averaging over a loosely
  structured space produces a less informative point.
- Look for class-specific features: wings for bird/airplane, shape for automobile vs truck.

In [ ]:
print('CTLS class centroids:')
fig = viz_ctls.centroid_decoding(
    max_samples=5000,
    title='CTLS — Class Centroids ("Universal" concepts per class)',
)
fig.savefig('experiments/decoder/centroids_ctls.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('Baseline class centroids:')
fig = viz_base.centroid_decoding(
    max_samples=5000,
    title='Baseline — Class Centroids (for comparison)',
)
fig.savefig('experiments/decoder/centroids_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Latent Space Interpolation

Linearly interpolate between two class centroids in circuit space and decode
each step. This visualises the *shared visual features* in the model's
internal ambiguity zone between two classes.

Selected pairs:
- **Dog ↔ Cat** — same superordinate category; should share fur/shape features
- **Automobile ↔ Truck** — known CTLS confusion pair (low intraclass ρ in Stage 3)
- **Bird ↔ Airplane** — often confused; both have 'wings' circuit activation
- **Dog ↔ Deer** — anatomically different; sharp transition expected

In [ ]:
# CIFAR-10 class indices:
# 0=airplane, 1=automobile, 2=bird, 3=cat, 4=deer
# 5=dog, 6=frog, 7=horse, 8=ship, 9=truck

pairs = [
    (5, 3, 'dog_cat'),         # dog ↔ cat
    (1, 9, 'automobile_truck'), # automobile ↔ truck
    (2, 0, 'bird_airplane'),   # bird ↔ airplane
    (5, 4, 'dog_deer'),        # dog ↔ deer
]

for class_a, class_b, fname in pairs:
    fig = viz_ctls.interpolation_grid(
        class_a=class_a, class_b=class_b,
        n_steps=10, max_samples=5000,
    )
    fig.savefig(f'experiments/decoder/interp_{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Confusion Zone Analysis

For each selected class pair, find real images from class A whose circuit
embedding is closest (cosine distance) to class B's centroid, then decode them.

These are the images where the model's *internal reasoning* most resembles
its reasoning for the other class — even if the final prediction was correct.
The decoded images reveal which visual features are driving the confusion.

**What to look for:** If the confusion is spurious (background, texture artifact),
the decoded images will show those spurious features prominently. If the confusion
is genuine (shared shape), the decoded images will show that shape.

This is the key diagnostic use-case for the circuit decoder.

In [ ]:
confusion_pairs = [
    (3, 5, 'cat→dog'),          # cat images closest to dog centroid
    (1, 9, 'automobile→truck'), # automobile images closest to truck centroid
    (4, 7, 'deer→horse'),       # deer images closest to horse centroid
]

for class_a, class_b, fname in confusion_pairs:
    fig = viz_ctls.confusion_zone(
        class_a=class_a, class_b=class_b,
        n_images=8, max_samples=5000,
    )
    fig.savefig(f'experiments/decoder/confusion_{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

## Summary

Fill in the results table:

| Metric | CTLS Decoder | Baseline Decoder | Delta |
|--------|-------------|-----------------|-------|
| Mean val MSE | ___ | ___ | ___ |
| Centroid quality (visual) | ___ | ___ | — |

**Interpretation guide:**
- **Lower MSE for CTLS** confirms that structured z encodes more recoverable
  visual information than unstructured z.
- **Sharper centroid images** for CTLS show that class clusters are tight and
  geometrically meaningful — not just labels attached to noise.
- **Interpolation smoothness:** CTLS interpolations should transition gradually
  through recognisable intermediate forms. Baseline interpolations may be abrupt
  or visually incoherent.
- **Confusion zone:** The decoded images identify *which visual features* the
  model's circuits actually use to confuse two classes. Spurious features
  (backgrounds, textures) vs. genuine shared shape — this is the diagnostic payoff.

**Next directions:**
- Idea 3 (Robustness Auditing): use the circuit decoder to visualise how
  noise perturbs the circuit trajectory — decode the noisy z to see what
  the model 'thinks' it's seeing under distribution shift.
- Idea 4 (Divergence Analysis): decode the trajectory *at the divergence
  point layer* to visualise where the model's concept breaks down.